In [1]:
import os
from pyspark.sql.functions import count, col, when, broadcast, udf, pandas_udf, rand, monotonically_increasing_id
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, PCA, StandardScalerModel, PCAModel
from pyspark.ml.clustering import KMeans, KMeansModel
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, BooleanType, DoubleType, ArrayType, IntegerType, StructType, StructField, FloatType
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from pyspark.ml.evaluation import ClusteringEvaluator
import datamol as dm
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
import logging
import matplotlib.pyplot as plt
import findspark
import os
import sys

In [2]:
findspark.init()


In [14]:
spark = SparkSession.builder\
    .appName("Molecule Analysis")\
    .config("spark.executor.memory", "16g")\
    .config("spark.driver.memory", "12g")\
    .config("spark.memory.fraction", "0.7")\
    .config("spark.memoary.storageFraction", "0.4")\
    .config("spark.network.timeout", "1200s")\
    .config("spark.executor.heartbeatInterval", "200s")\
    .config("spark.sql.broadcastTimeout", "1500s")\
    .getOrCreate()


25/02/04 17:47:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [13]:
filename = "leash-BELKA/train.parquet"
data = spark.read.parquet(filename)

data.printSchema()

data.show(5)

root
 |-- id: long (nullable = true)
 |-- buildingblock1_smiles: string (nullable = true)
 |-- buildingblock2_smiles: string (nullable = true)
 |-- buildingblock3_smiles: string (nullable = true)
 |-- molecule_smiles: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- binds: long (nullable = true)

+---+---------------------+---------------------+---------------------+--------------------+------------+-----+
| id|buildingblock1_smiles|buildingblock2_smiles|buildingblock3_smiles|     molecule_smiles|protein_name|binds|
+---+---------------------+---------------------+---------------------+--------------------+------------+-----+
|  0| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|        BRD4|    0|
|  1| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|         HSA|    0|
|  2| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|         sEH|    0|
|  3|

In [6]:
data = data.filter(col("protein_name") == "sEH")
sampled_data = data.orderBy(rand()).limit(100000)

In [7]:
desc_schema = StructType([
    StructField("mol_wt", DoubleType(), True),
    StructField("mol_logp", DoubleType(), True),
    StructField("tpsa", DoubleType(), True),
]
)

@pandas_udf(desc_schema)
def calculate_desc(smiles_series: pd.Series) -> pd.DataFrame:

    mols = smiles_series.apply(Chem.MolFromSmiles)
    
    mw = mols.apply(lambda mol: Descriptors.MolWt(mol) if mol else None)
    logp = mols.apply(lambda mol: Descriptors.MolLogP(mol) if mol else None)
    tpsa = mols.apply(lambda mol: Descriptors.TPSA(mol) if mol else None)

    return pd.DataFrame({'mol_wt': mw, 'mol_logp': logp, 'tpsa': tpsa})
    
sampled_data_with_desc = sampled_data.withColumn("molecule", calculate_desc(col("molecule_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block1", calculate_desc(col("buildingblock1_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block2", calculate_desc(col("buildingblock2_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block3", calculate_desc(col("buildingblock3_smiles")))

In [8]:
selected_columns = [
    "id",
    col("molecule.mol_wt").alias("molecule_mol_wt"),
    col("molecule.mol_logp").alias("molecule_mol_logp"),
    col("molecule.tpsa").alias("molecule_tpsa"),
    col("block1.mol_wt").alias("block1_mol_wt"),
    col("block1.mol_logp").alias("block1_mol_logp"),
    col("block1.tpsa").alias("block1_tpsa"),
    col("block2.mol_wt").alias("block2_mol_wt"),
    col("block2.mol_logp").alias("block2_mol_logp"),
    col("block2.tpsa").alias("block2_tpsa"),
    col("block3.mol_wt").alias("block3_mol_wt"),
    col("block3.mol_logp").alias("block3_mol_logp"),
    col("block3.tpsa").alias("block3_tpsa"),
    "binds"
]

flattened_data = sampled_data_with_desc.select(*selected_columns)

In [9]:
desc_columns = [col_name for col_name in flattened_data.columns if col_name != "id"]

assembler = VectorAssembler(inputCols=desc_columns, outputCol="features")
sampled_data_with_features = assembler.transform(flattened_data)

In [ ]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(sampled_data_with_features)
scaled_data = scaler_model.transform(sampled_data_with_features)

In [15]:
scaled_data.select("id", "scaled_features").write.save("./intermediates/scaled_data.parquet", format="parquet", mode='overwrite')
scaler_model.save("./intermediates/scaler_model")

In [23]:
scaler_model = StandardScalerModel.load("./intermediates/scaler_model")

In [24]:
scaled_data = scaler_model.transform(sampled_data_with_features)

In [ ]:
scaled_data.select("id", "scaled_features").write.save("./intermediates/scaled_data.parquet", format="parquet", mode='overwrite')

In [16]:
pca = PCA(k=8, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(scaled_data)
reduced_data = pca_model.transform(scaled_data)

25/02/04 19:24:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/02/04 19:25:49 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [17]:
reduced_data.select("pca_features").write.save("./intermediates/reduced_data.parquet", format="parquet", mode='overwrite')
pca_model.save("./intermediates/pca_model")

In [ ]:
reduced_data.select("id", "pca_features").write.save("./intermediates/reduced_data.parquet", format="parquet", mode='overwrite')
pca_model.save("./intermediates/pca_model")

In [114]:
spark.stop()

In [4]:
spark = SparkSession.builder\
    .appName("Molecule Analysis")\
    .config("spark.executor.memory", "22g")\
    .config("spark.driver.memory", "16g")\
    .config("spark.memory.fraction", "0.7")\
    .config("spark.memory.storageFraction", "0.4")\
    .config("spark.network.timeout", "1200s")\
    .config("spark.executor.heartbeatInterval", "200s")\
    .config("spark.sql.broadcastTimeout", "1500s")\
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()


In [5]:
filename = "leash-BELKA/train.parquet"
data = spark.read.parquet(filename)
data = data.filter(col("protein_name") == "sEH")
sampled_data = data.orderBy(rand()).limit(100000)

In [ ]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(sampled_data_with_features)
scaled_data = scaler_model.transform(sampled_data_with_features).select("id", "binds", "scaled_features")

In [103]:
scaler_model = StandardScalerModel.load("./intermediates/scaler_model")
scaled_data = scaler_model.transform(sampled_data_with_features)

In [104]:
pca_model = PCAModel.load("./intermediates/pca_model")
reduced_data = pca_model.transform(scaled_data)

In [115]:
kmeans_model = KMeansModel.load("./intermediates/kmeans_model")
kmeans_clusters = kmeans_model.transform(reduced_data).select("id", "binds", "kmeans_cluster")
kmeans_clusters.printSchema()

AssertionError: 

In [107]:
all_clustered_data = kmeans_clusters

for i in range(5):
    sampled_data = data.orderBy(rand()).limit(100000)
    sampled_data_with_desc = sampled_data.withColumn("molecule", calculate_desc(col("molecule_smiles")))
    sampled_data_with_desc = sampled_data_with_desc.withColumn("block1", calculate_desc(col("buildingblock1_smiles")))
    sampled_data_with_desc = sampled_data_with_desc.withColumn("block2", calculate_desc(col("buildingblock2_smiles")))
    sampled_data_with_desc = sampled_data_with_desc.withColumn("block3", calculate_desc(col("buildingblock3_smiles")))
    flattened_data = sampled_data_with_desc.select(*selected_columns)
    desc_columns = [col_name for col_name in flattened_data.columns if col_name != "id"]

    assembler = VectorAssembler(inputCols=desc_columns, outputCol="features")
    sampled_data_with_features = assembler.transform(flattened_data)

    scaled_data = scaler_model.transform(sampled_data_with_features)
    reduced_data = pca_model.transform(scaled_data)
    new_kmeans_clusters = kmeans_model.transform(reduced_data).select("id", "binds", "kmeans_cluster")

    all_clustered_data = all_clustered_data.union(new_kmeans_clusters)



In [108]:
all_clustered_data.printSchema()

root
 |-- id: long (nullable = true)
 |-- binds: long (nullable = true)
 |-- kmeans_cluster: integer (nullable = false)



In [109]:
binds1 = all_clustered_data.filter(col("binds") == 1)
binds0 = all_clustered_data.filter(col("binds") == 0)

In [112]:
binds_1_counts = binds1.groupBy("kmeans_cluster").count().withColumnRenamed("count", "count_binds_1")
binds_0_counts = binds0.groupBy("kmeans_cluster").count().withColumnRenamed("count", "count_binds_0")


cluster_counts = binds_1_counts.join(binds_0_counts, "kmeans_cluster", "inner")

In [113]:
fractions_df = cluster_counts.withColumn("fraction", (F.col("count_binds_1") * 3) / F.col("count_binds_0"))

fractions = fractions_df.select("kmeans_cluster", "fraction").rdd.collectAsMap()

25/02/07 00:26:08 ERROR Executor: Exception in task 4.0 in stage 64.0 (TID 1426)
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (StandardScalerModel$$$Lambda$5216/0x0000000841b8e040: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:217)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage46.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenExec$$anon$1.hasNext(WholeStageCodegenExec.scala:760)
	at org.apache.spark.sql.catalyst.expressions.GeneratedCla

Py4JJavaError: An error occurred while calling o4892.javaToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 4 in stage 64.0 failed 1 times, most recent failure: Lost task 4.0 in stage 64.0 (TID 1426) (10.202.33.187 executor driver): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (StandardScalerModel$$$Lambda$5216/0x0000000841b8e040: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:217)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage46.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(Buffered

In [ ]:
binds_0_sampled = binds0.sampleBy("kmeans_cluster", fractions, seed=42)

balanced_data = binds1.unionByName(binds_0_sampled.select(binds1.columns))

In [ ]:
balanced_data.write.save("./intermediates/balanced_data.parquet", format="parquet", mode='overwrite')

In [ ]:
'''
#VERY SLOW
ds1 = data_with_clusters.filter(col('binds') == 1)
binds0 = data_with_clusters.filter(col('binds') == 0)

binds_1_counts = binds1.groupBy("kmeans_cluster").count().withColumnRenamed("count", "count_binds_1")
binds_0_counts = binds0.groupBy("kmeans_cluster").count().withColumnRenamed("count", "count_binds_0")

binds_0_with_counts = binds0.join(binds_1_counts, on="kmeans_cluster", how="inner") \
    .join(binds_0_counts, on="kmeans_cluster", how="inner")

target_ratio = 3

binds_0_sampled = binds_0_with_counts.withColumn(
    "rand", rand()
).withColumn(
    "row_num", col("count_binds_1") * target_ratio 
).filter(col("rand") < col("row_num") / col("count_binds_0")) 

balanced_data = binds1.union(binds_0_sampled.select(binds1.columns))

balanced_data.groupBy("binds").count().show()
'''

In [ ]:
data_subset = data.select("id", "buildingblock1_smiles", "buildingblock2_smiles", "buildingblock3_smiles", "molecule_smiles")

final_data = balanced_data.drop("kmeans_cluster").join(data_subset, on="id", how="left")
final_data.write.save("./intermediates/final_data.parquet", format="parquet", mode='overwrite')

In [ ]:
from transformers import BertTokenizer, BertModel
import torch


tokenizer = BertTokenizer.from_pretrained('seyonec/PubChem10M_SMILES_BERT_Bio_110k')
model = BertModel.from_pretrained('seyonec/PubChem10M_SMILES_BERT_Bio_110k')

def smiles_embeddings(smiles:str):

    tokens = tokenizer(smiles, padding = True, truncation = True, max_length = 512, return_tensors = 'pt')
    
    with torch.no_grads():
        outputs = model(**tokens)
    
    return outputs.last_hidden_state.mean(dim = 1).squeeze().cpu().numpy().tolist()

balanced_data_with_embeddings = balanced_data.select(
    "buildingblock1_smiles", "buildingblock2_smiles", "buildingblock3_smiles", "molecule_smiles", "binds"
).toPandas()

balanced_data_with_embeddings['buildingblock1_embedding'] = balanced_data_with_embeddings['buildingblock1_smiles'].apply(smiles_embeddings)
balanced_data_with_embeddings['buildingblock2_embedding'] = balanced_data_with_embeddings['buildingblock2_smiles'].apply(smiles_embeddings)
balanced_data_with_embeddings['buildingblock3_embedding'] = balanced_data_with_embeddings['buildingblock3_smiles'].apply(smiles_embeddings)
balanced_data_with_embeddings['molecule_embedding'] = balanced_data_with_embeddings['molecule_smiles'].apply(smiles_embeddings)

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score


X = balanced_data_with_embeddings[['buildingblock1_embedding', 'buildingblock2_embedding', 'buildingblock3_embedding', 'molecule_embedding']]
y = balanced_data_with_embeddings['binds']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train.tolist(), y_train)  

y_pred = xgb_model.predict(X_test.tolist())
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}, F1: {f1:.4f}')